In [25]:
!pip install -q fastparquet
!pip install -q statsforecast

In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import TSB, AutoETS, AutoARIMA, Naive, SeasonalNaive, Theta, OptimizedTheta, CrostonOptimized, ADIDA, IMAPA, CrostonSBA

In [53]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

In [54]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'intermittent' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand

,unique_id,ds,y
15,140893170,2024-01-01,1.0
16,140893176,2024-01-01,1.0
17,140893177,2024-01-01,1.0
20,141242619,2024-01-01,1.0
39,150514660,2024-01-01,1.0
...,...,...,...
5265573,1839522281,2025-09-30,11.0
5265577,1995297405,2025-09-30,11.0
5265594,496029744,2025-09-30,12.0
5265614,1641858482,2025-09-30,12.0


In [55]:
# 1. Создаем список с моделями
models = [
    SeasonalNaive(season_length=7, alias='SeasonalNaive7'),    
    Naive(alias='Naive'),   
    CrostonOptimized(alias='CrostonOpt'),
    CrostonSBA(alias='CrostonSBA'),
    TSB(alpha_d=0.05, alpha_p=0.05, alias='TSB_005_005'),
    TSB(alpha_d=0.1, alpha_p=0.1, alias='TSB_01_01'),
    TSB(alpha_d=0.2, alpha_p=0.2, alias='TSB_02_02'),
    TSB(alpha_d=0.5, alpha_p=0.5, alias='TSB_05_05'),
    ADIDA(alias='ADIDA'),
    IMAPA(alias='IMAPA'),       
#    AutoETS(season_length=7, alias='AutoETS_seas7'),                                          
    OptimizedTheta(season_length=7, alias='OptTheta')  
]

# TSB - эталон для lumpy рядов
# ETS - семейство экспоненциальных сглаживаний
# SeasonalNaive - примитивный лаговый прогноз
# Naive - прогноз это последнее значение ряда
# Theta - разбивает ряд на компоненты (линии ряда), каждая компонента прогнозируется отдельно, потом прогнозы комбинируются для получения результата
# ARIMA - семейство ARIMA моделей

In [56]:
# задаем параметры
horizon = 14
step_size = 7
n_windows = 5

# определяем минимально необходимую длину ряда
min_required = n_windows * step_size + horizon

# Группировка и подсчёт длины
series_len = real_demand.groupby('unique_id').size()
long_series = series_len[series_len >= min_required].index

# Оставляем только длинные ряды
real_demand_filtered = real_demand[real_demand['unique_id'].isin(long_series)]

# Создаём экземпляр StatsForecast со списком моделей
sf = StatsForecast(models=models, freq='D', n_jobs=1)

# Теперь запускаем кросс-валидацию на отфильтрованных данных
cv_results = sf.cross_validation(
    df=real_demand_filtered,
    h=horizon,
    step_size=step_size,
    n_windows=n_windows
)

In [57]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -------------------------------
# 1. Определение метрик
# -------------------------------

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    denominator = np.abs(y_true) + np.abs(y_pred)
    mask = denominator > 0
    if mask.sum() == 0:
        return np.nan
    return (2 * np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]).mean() * 100

def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    denominator = np.abs(y_true).sum()
    if denominator == 0:
        return np.nan
    return np.abs(y_true - y_pred).sum() / denominator * 100

# -------------------------------
# 2. Функция для вычисления метрик по каждому окну (без unique_id)
# -------------------------------

def compute_metrics_per_window(cv_results, model_columns, metrics_dict):
    """
    cv_results: DataFrame от sf.cross_validation()
    model_columns: список имён столбцов с прогнозами моделей
    metrics_dict: словарь вида {'mae': mae, 'rmse': rmse, ...}
    Возвращает DataFrame с колонками: cutoff, model, и каждой метрикой.
    Для каждого cutoff (окна) метрика вычисляется по всем точкам всех unique_id.
    """
    records = []
    grouped = cv_results.groupby('cutoff')   # группируем только по окну
    
    for cutoff, group in grouped:
        y_true = group['y'].values
        for model in model_columns:
            y_pred = group[model].values
            row = {'cutoff': cutoff, 'model': model}
            for metric_name, metric_func in metrics_dict.items():
                row[metric_name] = metric_func(y_true, y_pred)
            records.append(row)
    
    return pd.DataFrame(records)

# -------------------------------
# 3. Применение к вашим данным
# -------------------------------

# Предполагаем, что cv_results уже получен после cross_validation
# Определяем столбцы моделей (все, кроме служебных)
model_columns = [col for col in cv_results.columns 
                 if col not in ['unique_id', 'ds', 'cutoff', 'y']]

metrics = {
    'mae': mae,
    'rmse': rmse,
    'smape': smape,
    'wape': wape
}

# Вычисляем метрики для каждого окна (без разбивки по unique_id)
metrics_per_window = compute_metrics_per_window(cv_results, model_columns, metrics)

# -------------------------------
# 4. Агрегация результатов
# -------------------------------

# 4.1. Среднее по всем окнам для каждой модели
summary_mean = metrics_per_window.groupby('model')[['mae', 'rmse', 'smape', 'wape']].mean()


# 4.2. Детальная статистика (mean, std, min, max) по окнам
summary_stats = metrics_per_window.groupby('model')[['mae', 'rmse', 'smape', 'wape']].agg(['mean', 'std', 'min', 'max'])


summary_mean

,mae,rmse,smape,wape
model,,,,
ADIDA,0.543718,0.970689,174.641993,119.977823
CrostonOpt,0.645485,1.051867,172.356451,152.987844
CrostonSBA,0.659168,1.058208,172.333941,157.374119
IMAPA,0.542977,0.972770,174.455539,119.573387
Naive,0.596687,1.224695,159.641199,129.150261
OptTheta,0.574006,1.026102,175.209986,126.040435
SeasonalNaive7,0.613419,1.255459,160.200029,132.866668
TSB_005_005,0.596774,1.013596,172.599134,137.514864
TSB_01_01,0.556756,0.974283,173.291960,125.343431
